In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASIC PATHS DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)
CHECKPOINT_DIR=os.path.join(
    BASE_DIR,
    "checkpoints"
)

print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

DATASET LOADING

In [ ]:
#load common ids of 529 videos
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

OUTPUT_DIR = os.path.join(BASE_DIR,"outputs")

common_df = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "common_samples.csv"
    )
)

common_ids = common_df[
    "video_id"
].tolist()

print("Common IDs:", len(common_ids))

In [ ]:
#load splits
import numpy as np

X_train = np.load(f"{OUTPUT_DIR}/X_train.npy")
X_val   = np.load(f"{OUTPUT_DIR}/X_val.npy")
X_test  = np.load(f"{OUTPUT_DIR}/X_test.npy")

Y_train = np.load(f"{OUTPUT_DIR}/Y_train.npy")
Y_val   = np.load(f"{OUTPUT_DIR}/Y_val.npy")
Y_test  = np.load(f"{OUTPUT_DIR}/Y_test.npy")

print(X_train.shape)
print(Y_train.shape)

In [ ]:
#dataset pytorch
import torch
from torch.utils.data import Dataset, DataLoader

class EmbeddingDataset(Dataset):

    def __init__(self, X, Y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.Y = torch.tensor(
            Y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.Y[idx]
        )

In [ ]:
#dataset loader
train_dataset = EmbeddingDataset(
    X_train,
    Y_train
)

val_dataset = EmbeddingDataset(
    X_val,
    Y_val
)

test_dataset = EmbeddingDataset(
    X_test,
    Y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

MLP CREATION

In [ ]:
#mlp model
import torch.nn as nn

class FusionMLP(nn.Module):

    def __init__(
        self,
        hidden_dim=1024,
        dropout_rate=0.2
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                640,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout_rate
            ),

            nn.Linear(
                hidden_dim,
                768
            )
        )

    def forward(self, x):

        return self.net(x)

In [ ]:
#hyperparameter grid
import pandas as pd

tuning_grid = [

    {
        "hidden": 512,
        "dropout": 0.2,
        "lr": 1e-4,
        "notes": "Low Capacity"
    },

    {
        "hidden": 1024,
        "dropout": 0.2,
        "lr": 1e-4,
        "notes": "Current Baseline"
    },

    {
        "hidden": 1024,
        "dropout": 0.0,
        "lr": 1e-4,
        "notes": "No Dropout"
    },

    {
        "hidden": 1024,
        "dropout": 0.5,
        "lr": 1e-4,
        "notes": "Heavy Dropout"
    },

    {
        "hidden": 2048,
        "dropout": 0.2,
        "lr": 1e-4,
        "notes": "High Capacity"
    },

    {
        "hidden": 1024,
        "dropout": 0.2,
        "lr": 1e-3,
        "notes": "High LR"
    }

]

In [ ]:
#gpu
#device
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
#model+loss
model = FusionMLP().to(device)

mse_loss = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
#cosine+mse loss(combined loss)
import torch.nn.functional as F

def total_loss(
    pred,
    target
):

    mse = mse_loss(
        pred,
        target
    )

    cosine = 1 - F.cosine_similarity(
        pred,
        target
    ).mean()

    return mse + cosine

TRAINING

In [ ]:
#for storing results
results_log = []

In [ ]:
#training
from tqdm import tqdm
import torch
import os

EPOCHS = 20

for trial in tuning_grid:

    print("\n")
    print("="*60)

    print(
        f"Hidden={trial['hidden']} "
        f"Dropout={trial['dropout']} "
        f"LR={trial['lr']}"
    )

    print(
        trial["notes"]
    )

    print("="*60)

    model = FusionMLP(

        hidden_dim=
        trial["hidden"],

        dropout_rate=
        trial["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=trial["lr"]

    )

    best_val = 999999

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0

        for x_batch, y_batch in train_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            pred = model(
                x_batch
            )

            loss = total_loss(
                pred,
                y_batch
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        train_loss = (

            running_loss
            /
            len(train_loader)

        )

        model.eval()

        running_val = 0

        with torch.no_grad():

            for x_batch, y_batch in val_loader:

                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(
                    x_batch
                )

                loss = total_loss(
                    pred,
                    y_batch
                )

                running_val += loss.item()

        val_loss = (

            running_val
            /
            len(val_loader)

        )

        print(

            f"Epoch {epoch+1} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f}"

        )

        if val_loss < best_val:

            best_val = val_loss

            best_model_path = os.path.join(

                CHECKPOINT_DIR,

                f"mlp_h{trial['hidden']}"
                f"_d{trial['dropout']}"
                f"_lr{trial['lr']}.pth"

            )

            torch.save(

                model.state_dict(),

                best_model_path

            )

    results_log.append({

        "Hidden":
        trial["hidden"],

        "Dropout":
        trial["dropout"],

        "LR":
        trial["lr"],

        "Description":
        trial["notes"],

        "Best Val Loss":
        round(best_val, 4)

    })

RESULTS OF DIFFRENT PARAMETERS COMBINATION

In [ ]:
#dataframe of results
df_results = pd.DataFrame(
    results_log
)

df_results

In [ ]:
#saving dataframe as csv to output directory
df_results.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "mlp_hyperparameter_results.csv"

    ),

    index=False

)

print("Saved")

FINDING BEST MODEL

In [ ]:
#finding best model
df_results = pd.DataFrame(results_log)

df_results = df_results.sort_values(
    "Best Val Loss"
)

df_results

In [ ]:
best_trial = df_results.iloc[0]

print(best_trial)

In [ ]:
#loading best model
best_path = os.path.join(

    CHECKPOINT_DIR,

    f"mlp_h{int(best_trial['Hidden'])}"
    f"_d{best_trial['Dropout']}"
    f"_lr{best_trial['LR']}.pth"

)

print(best_path)

In [ ]:
model = FusionMLP(

    hidden_dim=
    int(best_trial["Hidden"]),

    dropout_rate=
    float(best_trial["Dropout"])

).to(device)

In [ ]:
model.load_state_dict(

    torch.load(
        best_path,
        map_location=device
    )

)

model.eval()

print("Best model loaded")

PREDICTIONS FOR TEST SET

In [ ]:
#predictions for test set
import numpy as np

predictions = []
targets = []

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)

        pred = model(
            x_batch
        )

        predictions.append(

            pred.cpu().numpy()

        )

        targets.append(

            y_batch.numpy()

        )

predictions = np.concatenate(
    predictions,
    axis=0
)

targets = np.concatenate(
    targets,
    axis=0
)

print(predictions.shape)
print(targets.shape)

EVALUATION

In [ ]:
#mse
from sklearn.metrics import mean_squared_error

test_mse = mean_squared_error(

    targets,

    predictions

)

print(
    "Test MSE:",
    test_mse
)

In [ ]:
#cosine simlarity
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(

    predictions,

    targets

)

diag = np.diag(
    sim_matrix
)

print(
    "Mean Cosine Similarity:",
    diag.mean()
)

In [ ]:
#normalize embeddings
from sklearn.preprocessing import normalize

predictions = normalize(predictions)

targets = normalize(targets)

In [ ]:
#recall
ranks = []

for i in range(

    len(predictions)

):

    similarities = cosine_similarity(

        predictions[i].reshape(1,-1),

        targets

    )[0]

    ranking = np.argsort(

        similarities

    )[::-1]

    rank = np.where(

        ranking == i

    )[0][0] + 1

    ranks.append(rank)

ranks = np.array(ranks)

In [ ]:
R1 = np.mean(
    ranks <= 1
) * 100

R5 = np.mean(
    ranks <= 5
) * 100

R10 = np.mean(
    ranks <= 10
) * 100

MedR = np.median(
    ranks
)

print(
    f"Recall@1 : {R1:.2f}%"
)

print(
    f"Recall@5 : {R5:.2f}%"
)

print(
    f"Recall@10 : {R10:.2f}%"
)

print(
    f"Median Rank : {MedR}"
)

In [ ]:
#computation efficency -parameter count
params = sum(

    p.numel()

    for p in model.parameters()
)

print(
    "Parameters:",
    params
)

In [ ]:
#computation efficiency-inference time
import time

sample = torch.randn(
    1,
    640
).to(device)

start = time.time()

for _ in range(100):

    model(sample)

end = time.time()

print(
    "Average ms:",
    ((end-start)/100)*1000
)

In [ ]:
#simlarity marix
similarity_matrix = predictions @ targets.T

print(similarity_matrix.shape)

In [ ]:
print("Diagonal Mean:")

diag = np.mean(
    np.diag(
        similarity_matrix
    )
)

print(diag)

print("Overall Mean:")

print(
    similarity_matrix.mean()
)